# VGGT-1B on Kaggle: pretrained weights through the from-scratch implementation

Settings → **Accelerator: GPU T4 x2** (P100 may be unsupported by current PyTorch builds), **Internet: on** (needs a verified account), **Persistence: Files only**. Run Part 2 via *Save Version → Save & Run All (Commit)*.

1. Download the released `facebook/VGGT-1B` weights.
2. Run the **official** implementation and **this** implementation on the same real images and compare outputs (should agree to fp32 round-off).
3. Save depth maps / point cloud / cameras from our implementation, and time inference vs. number of frames (cf. paper Table 9).

In [ ]:
!pip -q install huggingface_hub einops safetensors
!git clone -q https://github.com/facebookresearch/vggt /kaggle/working/vggt_official
!git clone -q https://github.com/Platinum-Saber/VGGT /kaggle/working/vggt_ours
OFFICIAL, OURS = "/kaggle/working/vggt_official", "/kaggle/working/vggt_ours"

In [ ]:
import os; os.environ["HF_HOME"] = "/kaggle/working/hf_cache"  # keep the 5 GB weights with "Persistence: Files only"
import glob, json, sys, time, importlib, torch
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download("facebook/VGGT-1B", "model.pt")
device = "cuda"
images_paths = sorted(glob.glob(f"{OFFICIAL}/examples/kitchen/images/*"))[:8]
print(len(images_paths), torch.cuda.get_device_name())

In [ ]:
def import_pkg(root):
    for k in [k for k in sys.modules if k == "vggt" or k.startswith("vggt.")]:
        del sys.modules[k]
    sys.path[:] = [root] + [p for p in sys.path if p not in (OFFICIAL, OURS)]
    importlib.invalidate_caches()

# --- official model (fp32) ---
import_pkg(OFFICIAL)
from vggt.models.vggt import VGGT as OfficialVGGT
from vggt.utils.load_fn import load_and_preprocess_images
images = load_and_preprocess_images(images_paths).to(device)
ref = OfficialVGGT(); ref.load_state_dict(torch.load(ckpt, map_location="cpu")); ref = ref.to(device).eval()
with torch.no_grad():
    out_ref = {k: v.float().cpu() for k, v in ref(images).items() if k in ["pose_enc","depth","depth_conf","world_points","world_points_conf"]}
del ref; torch.cuda.empty_cache()

In [ ]:
# --- our model (fp32), same weights ---
import_pkg(OURS)
from vggt import VGGT
ours = VGGT.from_checkpoint(ckpt).to(device).eval()
with torch.no_grad():
    out_ours = {k: v.float().cpu() for k, v in ours(images).items() if k in out_ref}
for k in out_ref:
    d = (out_ref[k] - out_ours[k]).abs()
    print(f"{k:18s} max|diff|={d.max():.2e}  mean|diff|={d.mean():.2e}  max|ref|={out_ref[k].abs().max():.2e}")

In [ ]:
# --- runtime / memory vs #frames (paper Table 9 reports H100 numbers; T4 will be slower) ---
frames_all = sorted(glob.glob(f"{OFFICIAL}/examples/room/images/*")) + sorted(glob.glob(f"{OFFICIAL}/examples/kitchen/images/*"))
import_pkg(OFFICIAL); from vggt.utils.load_fn import load_and_preprocess_images; import_pkg(OURS)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
for n in [1, 2, 4, 8, 16, 32]:
    if n > len(frames_all): break
    x = load_and_preprocess_images(frames_all[:n]).to(device)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t = time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=dtype):
        ours(x)
    torch.cuda.synchronize()
    print(f"{n:3d} frames: {time.time()-t:6.2f}s  peak mem {torch.cuda.max_memory_allocated()/2**30:5.1f} GiB")

In [ ]:
# --- export depth / point cloud / cameras with our implementation ---
del ours; torch.cuda.empty_cache()
!python {OURS}/scripts/demo_pretrained.py --checkpoint {ckpt} --images {OFFICIAL}/examples/kitchen/images/* --out /kaggle/working/kitchen_out
!ls /kaggle/working/kitchen_out | head

## Part 2 — train the small VGGT for longer on the GPU

On the 4-core CPU sandbox each model got only 3k steps (~2.5 h), which was enough for depth / points but not for camera pose. Here the same scripts run ~10x longer. Renders the procedural data first (~7 min on CPU), then trains Alternating-Attention vs. global-only with matched parameter counts and evaluates both.

In [ ]:
%cd {OURS}
!python -m training.train --out runs/aa_long --aa-order frame global --depth 6 --steps 30000 --batch 16 --warmup 1000 2>&1 | grep -E '"step": [0-9]*000,|Error' 

In [ ]:
!python -m training.train --out runs/global_long --aa-order global --depth 12 --steps 30000 --batch 16 --warmup 1000 2>&1 | grep -E '"step": [0-9]*000,|Error' 

In [ ]:
!python -m training.evaluate runs/aa_long runs/global_long --frames 2 4 6 --out results/eval_long.json
!python -m training.baseline_sift --frames 4 --out results/eval_long.json